In [1]:

import torch
import nltk
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import f1_score, accuracy_score

nltk.download("punkt_tab", quiet=True)

if torch.backends.mps.is_available():
  device = torch.device('mps')

  print(device)

else:
  print("No GPU available, using the CPU instead")
  device = torch.device("cpu")

  

mps


## 1) Data Extraction and Sampling
Extract raw data, refresh destination folders, and build train/dev/test samples.

In [2]:
import os
import random
import shutil
from pathlib import Path

# Configuration
TOTAL_FILES_TO_SAMPLE = 10000
train_ratio = 0.70
dev_ratio = 0.10
test_ratio = 0.20

# Paths
local_base_path = 'dataset/canonical'
sampled_path = 'dataset/sampled_canonical'

if os.path.exists(sampled_path):
    print(f"Directory already exists: {sampled_path}. Skipping sampling/copying.")
else:
    # Create the directory structure
    os.makedirs(sampled_path, exist_ok=True)
    for split in ['train', 'dev', 'test']:
        os.makedirs(os.path.join(sampled_path, split), exist_ok=True)

    # Calculate counts
    train_count = int(TOTAL_FILES_TO_SAMPLE * train_ratio)
    dev_count = int(TOTAL_FILES_TO_SAMPLE * dev_ratio)
    test_count = TOTAL_FILES_TO_SAMPLE - train_count - dev_count  # Ensure it adds up exactly

    splits = {
        'train': train_count,
        'dev': dev_count,
        'test': test_count
    }

    print(f"Target distribution -> Train: {train_count}, Dev: {dev_count}, Test: {test_count}\n")

    for split, count in splits.items():
        src_dir = os.path.join(local_base_path, split)
        dst_dir = os.path.join(sampled_path, split)

        if os.path.exists(src_dir):
            # Get all json files in the source directory
            all_files = [f for f in os.listdir(src_dir) if f.endswith('.json')]
            print(f"Found {len(all_files)} files in local {split} folder.")

            # Sample files (or take all if count > available files)
            files_to_copy = random.sample(all_files, min(count, len(all_files)))

            print(f"Copying {len(files_to_copy)} files to Drive {split} folder...")
            for f in files_to_copy:
                shutil.copy2(os.path.join(src_dir, f), os.path.join(dst_dir, f))
            print(f"Finished copying {split}.\n")
        else:
            print(f"Warning: Source directory {src_dir} not found!")

    print("Sampling and copying process completed!")

Directory already exists: dataset/sampled_canonical. Skipping sampling/copying.


## 2) Data Sanity Check
Inspect one sample file to verify schema and content shape.

In [3]:
import os
import json

sample_dir = 'dataset/sampled_canonical/train'

if os.path.exists(sample_dir):
    sample_files = [f for f in os.listdir(sample_dir) if f.endswith('.json')]
    if sample_files:
        sample_file_path = os.path.join(sample_dir, sample_files[0])
        print(f"Reading sample file: {sample_file_path}\n")

        with open(sample_file_path, 'r') as f:
            data = json.load(f)

        print("JSON Keys:", list(data.keys()))
        print("\nFull JSON Content:")
        print(json.dumps(data, indent=2))
    else:
        print("No JSON files found in the directory.")
else:
    print(f"Directory not found: {sample_dir}")

Reading sample file: dataset/sampled_canonical/train/108928.json

JSON Keys: ['id', 'url', 'clean_article', 'clean_summary', 'extractive_summary']

Full JSON Content:
{
  "id": 108928,
  "url": "https://www.liputan6.com/news/read/108928/kejuaraan-jejak-lintas-alam-di-lereng-gunung-merapi",
  "clean_article": [
    [
      "Liputan6",
      ".",
      "com",
      ",",
      "Yogyakarta",
      ":",
      "Sekitar",
      "2",
      ".",
      "500",
      "orang",
      "mengikuti",
      "Kejuaraan",
      "Lacak",
      "Jejak",
      "Lintas",
      "Alam",
      "atau",
      "hash",
      "di",
      "lereng",
      "Gunung",
      "Merapi",
      ",",
      "Yogyakarta",
      ",",
      "Ahad",
      "(",
      "11/9",
      ")",
      "."
    ],
    [
      "Lomba",
      "yang",
      "dilepas",
      "Gubernur",
      "Yogyakarta",
      "Sri",
      "Sultan",
      "Hamengku",
      "Buwono",
      "X",
      "dan",
      "Ketua",
      "Hash",
      "Letnan",
      "Jendera

## 3) Load and Structure Dataset
Load JSON files into pandas DataFrames for each split.

In [4]:
import os
import json
import pandas as pd
from tqdm.notebook import tqdm

def load_data_split(split_path):
    records = []
    if not os.path.exists(split_path):
        print(f"Path not found: {split_path}")
        return pd.DataFrame()

    files = [f for f in os.listdir(split_path) if f.endswith('.json')]
    for f in tqdm(files, desc=f"Loading {os.path.basename(split_path)}"):
        with open(os.path.join(split_path, f), 'r') as file:
            data = json.load(file)

            # Form the full text from nested lists
            article = " ".join([" ".join(sentence) for sentence in data.get('clean_article', [])])
            summary = " ".join([" ".join(sentence) for sentence in data.get('clean_summary', [])])

            records.append({
                'id': data.get('id'),
                'url': data.get('url'),
                'article': article,
                'summary': summary,
                'extractive_summary': data.get('extractive_summary')
            })

    return pd.DataFrame(records)

# Base directory where your sampled data is stored
base_dir = 'dataset/sampled_canonical'

# Load the datasets
print("Loading datasets into Pandas DataFrames...")
df_train = load_data_split(os.path.join(base_dir, 'train'))
df_dev = load_data_split(os.path.join(base_dir, 'dev'))
df_test = load_data_split(os.path.join(base_dir, 'test'))

print("\nData Loading Complete!")
print(f"Train size: {len(df_train)}")
print(f"Dev size: {len(df_dev)}")
print(f"Test size: {len(df_test)}")

# Preview the train dataframe
display(df_train.head())

Loading datasets into Pandas DataFrames...


Loading train:   0%|          | 0/7000 [00:00<?, ?it/s]

Loading dev:   0%|          | 0/1000 [00:00<?, ?it/s]

Loading test:   0%|          | 0/2000 [00:00<?, ?it/s]


Data Loading Complete!
Train size: 7000
Dev size: 1000
Test size: 2000


,id,url,article,summary,extractive_summary
0,108928,https://www.liputan6.com/news/read/108928/keju...,"Liputan6 . com , Yogyakarta : Sekitar 2 . 500 ...",Kejuaraan Lacak Jejak Lintas Alam digelar di l...,"[0, 5]"
1,206060,https://www.liputan6.com/news/read/206060/opti...,Sosok yang telah lama ditunggu akhirnya mucul ...,Kebahagiaan merona di pipi Roman Abramovich . ...,"[0, 5, 2]"
2,81359,https://www.liputan6.com/news/read/81359/isu-k...,"Liputan6 . com , Pontianak : Sekelompok warga ...",Sekelompok warga melempari sebuah gereja di ko...,"[0, 4]"
3,89339,https://www.liputan6.com/news/read/89339/dpd-m...,"Liputan6 . com , Jakarta : Tim Kerja Dewan Pim...",Tim Kerja DPD yang bertugas membahas revisi UU...,"[0, 4]"
4,154246,https://www.liputan6.com/news/read/154246/musi...,"Liputan6 . com , Jakarta : Memasuki pancaroba ...",Pasien demam berdarah di Jakarta terus bertamb...,"[4, 8, 7]"


## 4) Tokenization and Preprocessing
Convert DataFrames to Hugging Face datasets and tokenize inputs/targets.

In [5]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2"], use_stemmer=False)


def greedy_oracle_labels(article_text: str, summary_text: str, top_k: int = 5):
    """
    Greedily select article sentences that maximize ROUGE with the reference summary.
    Returns list of sentences and 0/1 labels.
    """
    article_sents = nltk.sent_tokenize(article_text)
    if not article_sents:
        return [], []

    scores = []
    for sent in article_sents:
        r = scorer.score(summary_text, sent)
        combined = r["rouge1"].fmeasure + r["rouge2"].fmeasure
        scores.append(combined)

    top_indices = set(sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k])
    labels = [1 if i in top_indices else 0 for i in range(len(article_sents))]
    return article_sents, labels


def build_sentence_dataset(df):
    """Convert article-level DataFrame to sentence-level records."""
    records = {"sentence": [], "label": [], "article_id": []}
    for idx, row in df.iterrows():
        sents, labels = greedy_oracle_labels(row["article"], row["summary"])
        for s, l in zip(sents, labels):
            records["sentence"].append(s)
            records["label"].append(l)
            records["article_id"].append(idx)
    return Dataset.from_dict(records)


print("Building sentence-level datasets...")
sent_datasets = DatasetDict({
    "train": build_sentence_dataset(df_train),
    "validation": build_sentence_dataset(df_dev),
    "test": build_sentence_dataset(df_test),
})
print(f"Train: {len(sent_datasets['train'])} sentences")
print(f"Val:   {len(sent_datasets['validation'])} sentences")
print(f"Test:  {len(sent_datasets['test'])} sentences")

train_labels = sent_datasets["train"]["label"]
print(f"\nLabel distribution (train): 0={train_labels.count(0)}, 1={train_labels.count(1)}")

Building sentence-level datasets...
Train: 105258 sentences
Val:   15020 sentences
Test:  28198 sentences

Label distribution (train): 0=70264, 1=34994


In [6]:
model_checkpoint = "indolem/indobert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

MAX_LEN = 128  # per-sentence max length


def tokenize_fn(examples):
    return tokenizer(
        examples["sentence"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )


print("Tokenizing sentence datasets...")
tokenized_datasets = sent_datasets.map(tokenize_fn, batched=True, remove_columns=["sentence", "article_id"])
tokenized_datasets.set_format("torch")
print("Done! Features:", tokenized_datasets["train"].column_names)

/Users/fepriyadi/Documents/NLP/indonesian ai/Indonesia AI/Advance NLP/adv-nlp/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Tokenizing sentence datasets...


Map:   0%|          | 0/105258 [00:00<?, ? examples/s]

Map:   0%|          | 0/15020 [00:00<?, ? examples/s]

Map:   0%|          | 0/28198 [00:00<?, ? examples/s]

Done! Features: ['label', 'input_ids', 'token_type_ids', 'attention_mask']


## 5) Model and Trainer Setup
Initialize the seq2seq model, metrics, and training arguments.

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=2,
)
model.config.bos_token_id = tokenizer.bos_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.to(device)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(model.device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="binary"),
    }

# 4. Training Arguments

args = TrainingArguments(
    output_dir="./indolem-indobert-finetuned-out",
    # eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3, 
    gradient_checkpointing=True,
    weight_decay=0.01,
    save_total_limit=3,
    logging_steps=100,
    fp16=False,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


print("Trainer setup complete! Ready to start training.")

/Users/fepriyadi/Documents/NLP/indonesian ai/Indonesia AI/Advance NLP/adv-nlp/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indolem/indobert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


mps:0
Trainer setup complete! Ready to start training.


## 7) Training and Evaluation Run
Use the trainer in the previous cell to start fine-tuning and then evaluate on the test split.

Suggested next commands:
- `trainer.train()`
- `trainer.evaluate(tokenized_datasets["test"])`

In [10]:
trainer.train()

  0%|          | 0/9870 [00:00<?, ?it/s]

{'loss': 0.5783, 'learning_rate': 1.9797365754812566e-05, 'epoch': 0.03}
{'loss': 0.5458, 'learning_rate': 1.959473150962513e-05, 'epoch': 0.06}
{'loss': 0.5305, 'learning_rate': 1.939209726443769e-05, 'epoch': 0.09}
{'loss': 0.5356, 'learning_rate': 1.9189463019250255e-05, 'epoch': 0.12}
{'loss': 0.5228, 'learning_rate': 1.8986828774062816e-05, 'epoch': 0.15}
{'loss': 0.5336, 'learning_rate': 1.878419452887538e-05, 'epoch': 0.18}
{'loss': 0.5318, 'learning_rate': 1.8581560283687945e-05, 'epoch': 0.21}
{'loss': 0.5267, 'learning_rate': 1.837892603850051e-05, 'epoch': 0.24}
{'loss': 0.5263, 'learning_rate': 1.8176291793313074e-05, 'epoch': 0.27}
{'loss': 0.5149, 'learning_rate': 1.7973657548125635e-05, 'epoch': 0.3}
{'loss': 0.5207, 'learning_rate': 1.77710233029382e-05, 'epoch': 0.33}
{'loss': 0.5251, 'learning_rate': 1.756838905775076e-05, 'epoch': 0.36}
{'loss': 0.5257, 'learning_rate': 1.7365754812563324e-05, 'epoch': 0.4}
{'loss': 0.5244, 'learning_rate': 1.716312056737589e-05, 'ep

TrainOutput(global_step=9870, training_loss=0.487965007345244, metrics={'train_runtime': 11175.4702, 'train_samples_per_second': 28.256, 'train_steps_per_second': 0.883, 'train_loss': 0.487965007345244, 'epoch': 3.0})

In [ ]:
trainer.save_model('./indolem-indobert-finetuned/v1')

In [14]:
print(trainer.model.device)

mps:0


In [15]:
trainer.evaluate(tokenized_datasets["test"])

  0%|          | 0/441 [00:00<?, ?it/s]

{'eval_loss': 0.5331190228462219,
 'eval_accuracy': 0.7151925668487127,
 'eval_f1': 0.5896898789148317,
 'eval_runtime': 209.418,
 'eval_samples_per_second': 134.649,
 'eval_steps_per_second': 2.106,
 'epoch': 3.0}

In [ ]:
from tqdm import tqdm

# Batch inference on test dataset
def generate_summaries(model, tokenizer, articles, device, batch_size=8, top_k=5):
    summaries = []
    
    for i in tqdm(range(0, len(articles), batch_size), desc="Generating summaries"):
        batch_articles = articles[i:i+batch_size]
        
        # Process each article in the batch
        batch_summaries = []
        for article in batch_articles:
            # Split article into sentences
            sents = nltk.sent_tokenize(article)
            if not sents:
                batch_summaries.append("")
                continue
            
            # Tokenize all sentences in this article
            inputs = tokenizer(sents, truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # Get classification scores
            with torch.no_grad():
                logits = model(**inputs).logits
            
            # Get probabilities for class 1 (important sentence)
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            
            # Select top-k sentences by probability
            top_indices = sorted(range(len(probs)), key=lambda i: probs[i], reverse=True)[:top_k]
            top_indices = sorted(top_indices)  # Sort to maintain original order
            
            # Create summary from selected sentences
            summary = " ".join(sents[i] for i in top_indices)
            # print(f"\Summary:\n{summary}\n")
            batch_summaries.append(summary)
        
        summaries.extend(batch_summaries)
    
    return summaries

# Generate summaries for test set
# Load the saved model and tokenizer
model_path = './indolem-indobert-finetunedv2/'

print(f"Loading model from {model_path}...")
loaded_model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=2)
loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)

# Move model to device (GPU/CPU)
loaded_model.to(device)
loaded_model.eval()

print("Model and tokenizer loaded successfully!")
test_articles = df_test['article'].tolist()[:100]  # Using first 100 for testing
generated_summaries = generate_summaries(loaded_model, loaded_tokenizer, test_articles, device)

# Compare with actual summaries
results_df = pd.DataFrame({
    'article': test_articles,
    'original_summary': df_test['summary'].tolist()[:100],
    'generated_summary': generated_summaries
})

print("\n--- SAMPLE RESULTS ---")

# Configure pandas display options for better readability
pd.set_option('display.max_colwidth', None)  # Show full text in columns
pd.set_option('display.max_rows', 10)       # Show 10 rows
pd.set_option('display.width', None)        # Don't limit display width

# Display the results
display(results_df.head(10))

# Reset options if needed for other cells
# pd.reset_option('display.max_colwidth')

Loading model from ./indolem-indobert-finetunedv2/...
Model and tokenizer loaded successfully!


Generating summaries: 100%|██████████| 13/13 [00:05<00:00,  2.46it/s]


--- SAMPLE RESULTS ---


,article,original_summary,generated_summary
0,"Liputan6 . com , Ambon : Bahan bakar minyak jenis solar dan premium , selama sepekan terakhir , mulai langka di Kota Ambon . Akibatnya , sebagian besar kendaraan umum di Ambon memilih berhenti beroperasi karena sulit mendapatkan bahan bakar . Selain itu , kelangkaan tersebut juga memicu kenaikan harga premium di tingkat pengecer : mencapai Rp 2 ribu per liter . Padahal , biasanya harga premium hanya Rp 1 . 700 per liter . Demikian hasil pantauan SCTV di Ambon , baru-baru ini . Beberapa pedagang minyak eceran mengaku , kelangkaan terjadi karena pasokan dari Depot Pertamina Wayame berkurang . Selain itu , mereka juga menduga kelangkaan terjadi lantaran adanya ulah oknum yang menyuplai minyak ke sejumlah kapal di Pelabuhan Maluku . Menanggapi itu , Kapala Cabang Pertamina Unit Pemasaran dan Perbekalan Dalam Negeri VIII Ambon Subaedi memastikan , kelangkaan terjadi karena adanya kebijakan Pertamina Pusat yang mengurangi jatah pasokan . Subaedi memperkirakan , kelangkaan BBM di Ambon akan terus berlangsung hingga bulan depan . Tapi , ia menolak jika kelangkaan terjadi akibat adanya kerja sama kotor antara Pertamina dan para spekulan . ( ICH/Sahlan Heluth ) .","Sepekan terakhir , bahan bakar minyak jenis solar dan premium di Kota Ambon , mulai langka . Kelangkaan terjadi karena pasokan BBM dari Depot Pertamina Wayame berkurang .","com , Ambon : Bahan bakar minyak jenis solar dan premium , selama sepekan terakhir , mulai langka di Kota Ambon . Akibatnya , sebagian besar kendaraan umum di Ambon memilih berhenti beroperasi karena sulit mendapatkan bahan bakar . Selain itu , kelangkaan tersebut juga memicu kenaikan harga premium di tingkat pengecer : mencapai Rp 2 ribu per liter . Selain itu , mereka juga menduga kelangkaan terjadi lantaran adanya ulah oknum yang menyuplai minyak ke sejumlah kapal di Pelabuhan Maluku . Subaedi memperkirakan , kelangkaan BBM di Ambon akan terus berlangsung hingga bulan depan ."
1,"Liputan6 . com , Jakarta : Seluruh perubahan pasal Undang-Undang Nomor 23 tahun 1999 tentang Bank Indonesia telah disetujui oleh pemerintah dan DPR . Namun masih ada satu polemik soal pasal 75 . Inti pasal itu adalah soal memberhentikan seluruh anggota Dewan Gubernur BI , setelah UU Amendemen BI berlaku efektif . Bagi BI sendiri , posisinya harus tetap independen . Pernyataan itu ditegaskan Gubernur BI Sjahril Sabirin , baru-baru ini , di Jakarta . Menurut Sjahril , persoalan amendemen sudah menjadi kewenangan pemerintah dan DPR . Sebab kedua lembaga tersebut telah mendapat masukan dari tim panel Dana Moneter Internasional ( IMF ) . "" IMF sudah menganjurkan pembentukan tim panel , "" kata Sjahril . Lantas , tambah dia , tim panel itu juga sudah mengeluarkan hasil keputusan . "" Sekarang , tergantung pemerintah dan DPR untuk menerapkannya , "" kata Sjahril , serius . Pembahasan pasal 75 sendiri masih berlangsung di Panitia Kerja DPR . Panja ini mengusulkan pilihan supaya pemerintah mempercepat pembentukan Dewan Supervisi BI . Lantas , panja juga menawarkan kepada pemerintah untuk membuat laporan mengenai penyimpangan di tubuh BI . Berdasarkan laporan ini , DPR mesti menggelar paripurna untuk menentukan perlu tidaknya mengganti Gubernur BI . ( COK/Olivia Rosalia dan Bambang Triono ) .","Bank Indonesia menyerahkan sepenuhnya amendemen UU BI tentang pergantian dewan gubernur kepada pemerintah dan DPR . Diperkirakan , polemik soal pasal 75 sulit untuk bisa mencapai kata sepakat .","com , Jakarta : Seluruh perubahan pasal Undang-Undang Nomor 23 tahun 1999 tentang Bank Indonesia telah disetujui oleh pemerintah dan DPR . Inti pasal itu adalah soal memberhentikan seluruh anggota Dewan Gubernur BI , setelah UU Amendemen BI berlaku efektif . Pernyataan itu ditegaskan Gubernur BI Sjahril Sabirin , baru-baru ini , di Jakarta . Lantas , panja juga menawarkan kepada pemerintah untuk membuat laporan mengenai penyimpangan di tubuh BI . Berdasarkan laporan ini , DPR mesti menggelar paripurna untuk 

## Compute ROUGE scores on generated summaries

In [34]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge_scores = []
for _, row in results_df.iterrows():
    score = scorer.score(row['original_summary'], row['generated_summary'])
    rouge_scores.append({
        'rouge1': score['rouge1'].fmeasure,
        'rouge2': score['rouge2'].fmeasure,
        'rougeL': score['rougeL'].fmeasure
    })

rouge_df = pd.DataFrame(rouge_scores)
print("\n--- ROUGE SCORES (Average) ---")
print(rouge_df.mean())


--- ROUGE SCORES (Average) ---
rouge1    0.282789
rouge2    0.147772
rougeL    0.216847
dtype: float64


## 8) Load Saved Model and Perform Inference
Load the finetuned model from disk and use it for inference on new texts.